# University library help desk: queue simulation (SimPy)

The same code as `uni_library_simpy.py` (the version to run on your computer). The question is in `full-revision-booklet.pdf` and the written answer in `full-revision-booklet-solutions.pdf`.

Run the first cell, then **Run → Run All Cells**.

**What you should see:** with 2 advisers in the normal period about 95% of students wait under 8 minutes; in the peak period, on this single run, 3 advisers bring the maximum wait down to about 3 minutes (100% under 8). One run is not enough to decide: see the note at the end.

In [ ]:
# Run this cell first.
# On the course website (JupyterLite) it installs this notebook's packages into the browser tab
# (about a minute the first time) and adapts them to the browser.
# On your own computer it does nothing: install everything once with
#     pip install -r requirements.txt      (the file is at the top of the Assignments folder)
# In Google Colab, run  !pip install simpy  instead.
import sys
if sys.platform == "emscripten":  # only in the browser
    import micropip
    await micropip.install(["simpy==4.1.2"])
print("Ready.")

## Model

Two simulation functions. `run_helpdesk_simulation` generates students arriving at random (exponential gaps), queueing for one of the advisers and being served for a random time; it records each student's wait. `run_priority_helpdesk` does the same with a priority queue for urgent students. The seed fixes the random numbers, so every run gives the same output. This cell only defines the functions.

In [ ]:
from __future__ import annotations

import random

import pandas as pd
import simpy


def run_helpdesk_simulation(arrival_rate_per_hour: float, mean_service_minutes: float, advisers: int, run_minutes: int, seed: int, target_wait: float):
    rng = random.Random(seed)
    wait_times = []

    def arrivals(env, desk):
        student_number = 0
        while True:
            interarrival = rng.expovariate(arrival_rate_per_hour / 60.0)  # minutes until the next student
            yield env.timeout(interarrival)
            if env.now > run_minutes:
                break
            student_number += 1
            env.process(student(env, f"S{student_number}", desk))

    def student(env, name, desk):
        del name
        arrival_time = env.now
        with desk.request() as request:
            yield request
            wait = env.now - arrival_time
            wait_times.append(wait)
            service_time = rng.expovariate(1.0 / mean_service_minutes)  # random service time (minutes)
            yield env.timeout(service_time)

    env = simpy.Environment()
    desk = simpy.Resource(env, capacity=advisers)
    env.process(arrivals(env, desk))
    env.run(until=run_minutes + 10000)

    served = len(wait_times)
    return {
        "served": served,
        "average_wait": sum(wait_times) / served if served else 0.0,
        "maximum_wait": max(wait_times) if served else 0.0,
        "percent_under_target": 100 * sum(wait < target_wait for wait in wait_times) / served if served else 0.0,
    }


def run_priority_helpdesk(arrival_rate_per_hour: float, mean_service_minutes: float, advisers: int, run_minutes: int, seed: int, priority_share: float, target_wait: float):
    rng = random.Random(seed)
    results = []

    def arrivals(env, desk):
        student_number = 0
        while True:
            interarrival = rng.expovariate(arrival_rate_per_hour / 60.0)  # minutes until the next student
            yield env.timeout(interarrival)
            if env.now > run_minutes:
                break
            student_number += 1
            is_priority = rng.random() < priority_share
            env.process(student(env, f"S{student_number}", desk, is_priority))

    def student(env, name, desk, is_priority):
        arrival_time = env.now
        priority_level = 0 if is_priority else 1
        with desk.request(priority=priority_level) as request:
            yield request
            wait = env.now - arrival_time
            results.append({"student": name, "priority": is_priority, "wait": wait})
            service_time = rng.expovariate(1.0 / mean_service_minutes)  # random service time (minutes)
            yield env.timeout(service_time)

    env = simpy.Environment()
    desk = simpy.PriorityResource(env, capacity=advisers)
    env.process(arrivals(env, desk))
    env.run(until=run_minutes + 10000)

    output = pd.DataFrame(results)
    standard = output[output["priority"] == False]
    priority = output[output["priority"] == True]
    return {
        "served": int(len(output)),
        "overall_percent_under_target": float(100 * (output["wait"] < target_wait).mean()) if len(output) else 0.0,
        "priority_average_wait": float(priority["wait"].mean()) if len(priority) else 0.0,
        "standard_average_wait": float(standard["wait"].mean()) if len(standard) else 0.0,
    }


def main():
    baseline = run_helpdesk_simulation(10, 7, 2, 360, 24, 8)
    print("Baseline with 2 advisers:", baseline)

    rows = []
    for advisers in (2, 3, 4):
        result = run_helpdesk_simulation(15, 7, advisers, 360, 24, 8)
        rows.append(
            {
                "advisers": advisers,
                "served": result["served"],
                "avg_wait": round(result["average_wait"], 4),
                "max_wait": round(result["maximum_wait"], 4),
                "pct_below_8": round(result["percent_under_target"], 4),
            }
        )
    print("\nPeak-period table:")
    print(pd.DataFrame(rows).to_string(index=False))

    priority_result = run_priority_helpdesk(15, 7, 3, 360, 24, 0.10, 8)
    print("\nPriority extension with 3 advisers:", priority_result)

## Run the simulations

Runs the normal period with 2 advisers, the peak period with 2, 3 and 4 advisers, and the priority extension.

In [ ]:
main()

### What do you notice?

In the peak table the run with 3 advisers serves 73 students, but the runs with 2 and 4 advisers serve 70. The arrival rate and the seed are the same. Why do the numbers differ?

<details><summary>Answer</summary>

Arrivals and service times are drawn from **one** random stream. With a different number of advisers the draws are used in a different order, so each run sees a different sequence of arrivals. One run per option is therefore not a like-for-like comparison: use separate random streams (common random numbers) and many replications with confidence intervals before choosing. This seed also happens to give fewer arrivals than the average, so one run can make the desk look better staffed than it is.
</details>